Simple notebook for plotting morphologies using `dlutils` (available [here](https://github.com/danielelinaro/neuronopt.git)).

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from dlutils.morpho import Tree
from dlutils.graphics import plot_tree

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import CenteredNorm, TwoSlopeNorm
import seaborn as sns

fontsize = 9
lw = 0.75
matplotlib.rc('font', **{'family': 'Arial', 'size': fontsize})
matplotlib.rc('axes', **{'linewidth': 0.75, 'labelsize': fontsize})
matplotlib.rc('xtick', **{'labelsize': fontsize})
matplotlib.rc('ytick', **{'labelsize': fontsize})
matplotlib.rc('xtick.major', **{'width': lw, 'size':3})
matplotlib.rc('ytick.major', **{'width': lw, 'size':3})
matplotlib.rc('ytick.minor', **{'width': lw, 'size':1.5})

In [ ]:
project = 'dbbs-lab'
if project == 'mouselight':
    data_dir = os.path.join('..','morphologies','mouselight','CNG version')
elif project == 'dbbs-lab':
    data_dir = os.path.join('..','..','dbbs-lab','models','dbbs_models','morphologies')
else:
    raise Exception(f"Unknown project '{project}'")
swc_files = sorted(glob.glob(os.path.join(data_dir, '*.swc')))
if project == 'mouselight':
    swc_file = swc_files[6]
else:
    swc_file = swc_files[2]
tree = Tree(swc_file)
morpho = pd.read_table(swc_file, sep=' ', header=None, skiprows=3,
                       names=['type','x','y','z','diam','parent'],
                       skipinitialspace=True, index_col=0)

In [ ]:
bounds = tree.bounds
dx,dy = 10,10
x = np.r_[bounds[0,0] : bounds[0,1] : dx]
y = np.r_[bounds[1,0] : bounds[1,1] : dy]
X,Y = np.meshgrid(x, y, indexing='ij')
z = np.median(morpho.loc[:,'z'])

In [ ]:
height = 5
if project == 'mouselight':
    width = height * tree.xy_ratio
    coords = 'xy'
else:
    width = height * max(0.2, tree.xz_ratio)
    coords = 'xz'
fig,ax = plt.subplots(1, 1, figsize=(width,height))
plot_tree(tree, coords, ax=ax, diam_coeff=10, cmap=lambda _: [0,0,0])
ax.plot(tree.bounds[0,1]*1.1*np.ones(2), [0,50], 'k', lw=1)
ax.text(tree.bounds[0,1]+3, 25, '50 μm', rotation=90, fontsize=fontsize+2,
        horizontalalignment='left', verticalalignment='center')

ax.axis('off')
fig.tight_layout()
plt.savefig(os.path.splitext(os.path.basename(swc_file))[0] + '.pdf')